## 0. Imports

In [1]:
import pandas as pd
import numpy as np
import requests
import joblib
import time
import os

from itertools import product
from functools import partial
from dotenv import load_dotenv
from bs4 import BeautifulSoup
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyRegressor, DummyClassifier
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import (
    RandomForestRegressor,
    GradientBoostingRegressor,
    VotingRegressor,
    StackingRegressor,
    BaggingRegressor,
    RandomForestClassifier,
    GradientBoostingClassifier,
    VotingClassifier,
    StackingClassifier,
    BaggingClassifier
)
from sklearn.metrics import (
    mean_squared_error,
    accuracy_score,
    precision_score,
    make_scorer
)

## 1. Forecast

### a. Data Preparation

Загружаем данные

In [2]:
df = pd.read_csv("./data/epi_r.csv")

df

,title,rating,calories,protein,fat,sodium,#cakeweek,#wasteless,22-minute meals,3-ingredient recipes,...,yellow squash,yogurt,yonkers,yuca,zucchini,cookbooks,leftovers,snack,snack week,turkey
0,"Lentil, Apple, and Turkey Wrap",2.500,426.0,30.0,7.0,559.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
1,Boudin Blanc Terrine with Red Onion Confit,4.375,403.0,18.0,23.0,1439.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,Potato and Fennel Soup Hodge,3.750,165.0,6.0,7.0,165.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,Mahi-Mahi in Tomato Olive Sauce,5.000,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,Spinach Noodle Casserole,3.125,547.0,20.0,32.0,452.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20047,Parmesan Puffs,3.125,28.0,2.0,2.0,64.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
20048,Artichoke and Parmesan Risotto,4.375,671.0,22.0,28.0,583.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
20049,Turkey Cream Puff Pie,4.375,563.0,31.0,38.0,652.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
20050,Snapper on Angel Hair with Citrus Cream,4.375,631.0,45.0,24.0,517.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


Здесь находится полный список ингредиентов

In [3]:
full_list_of_ingredients = [
    "almond", "amaretto", "anchovy", "anise", "apple", "apple juice",
    "apricot", "artichoke", "arugula", "asian pear", "asparagus", "avocado",
    "bacon", "banana", "barley", "basil", "bass", "bean", "beef", "beef rib",
    "beef shank", "beef tenderloin", "beer", "beet", "bell pepper", "biscuit",
    "bitters", "blackberry", "blue cheese", "blueberry", "bok choy", "bourbon",
    "bran", "brandy", "bread", "breadcrumbs", "brie", "brine", "brisket",
    "broccoli", "broccoli rabe", "brown rice", "brownie", "brussel sprout",
    "buffalo", "bulgur", "burrito", "butter", "buttermilk", "butternut squash",
    "butterscotch/caramel", "cabbage", "calvados", "campari", "candy",
    "cantaloupe", "capers", "caraway", "cardamom", "carrot", "cashew",
    "cauliflower", "caviar", "celery", "chambord", "champagne", "chard",
    "chartreuse", "cheddar", "cheese", "cherry", "chestnut", "chicken",
    "chickpea", "chile", "chile pepper", "chive", "chocolate", "cilantro",
    "cinnamon", "clam", "clove", "coconut", "cod", "coffee", "cognac/armagnac",
    "collard greens", "cookie", "coriander", "corn", "cornmeal",
    "cottage cheese", "couscous", "crab", "cranberry", "cranberry sauce",
    "cream cheese", "créme de cacao", "cr��me de cacao", "cucumber", "cumin",
    "currant", "curry", "custard", "date", "dill", "duck", "eau de vie", "egg",
    "eggplant", "endive", "escarole", "fennel", "feta", "fig", "fish",
    "flat bread", "fontina", "fortified wine", "frangelico", "garlic", "gin",
    "ginger", "goat cheese", "goose", "gouda", "grand marnier", "granola",
    "grape", "grapefruit", "grappa", "green bean", "green onion/scallion",
    "ground beef", "ground lamb", "guava", "halibut", "ham", "hazelnut", 
    "hominy/cornmeal/masa", "honey", "honeydew", "horseradish", "hot pepper",
    "ice cream", "jalapeño", "jam or jelly", "jerusalem artichoke", "jícama",
    "kahlúa", "kale", "kirsch", "kiwi", "kumquat", "lamb", "lamb chop",
    "lamb shank", "leafy green", "leek", "lemon", "lemon juice", "lemongrass",
    "lentil", "lettuce", "lima bean", "lime", "lime juice", "lingonberry",
    "liqueur", "lobster", "lychee", "macadamia nut", "mango", "maple syrup",
    "marsala", "marscarpone", "marshmallow", "martini", "mayonnaise", "melon",
    "mezcal", "midori", "milk/cream", "mint", "molasses", "monterey jack",
    "mozzarella", "muffin", "mushroom", "mussel", "mustard", "mustard greens",
    "nectarine", "noodle", "nutmeg", "oat", "oatmeal", "octopus", "okra",
    "olive", "onion", "orange", "orange juice", "oregano", "orzo", "oyster",
    "pancake", "papaya", "paprika", "parmesan", "parsley", "parsnip",
    "passion fruit", "pasta", "pastry", "pea", "peach", "peanut",
    "peanut butter", "pear", "pecan", "pepper", "pernod", "persimmon",
    "phyllo/puff pastry dough", "pickles", "pie", "pine nut", "pineapple",
    "pistachio", "plantain", "plum", "poblano", "pomegranate",
    "pomegranate juice", "poppy", "pork", "pork chop", "pork rib",
    "pork tenderloin", "port", "potato", "poultry sausage", "prosciutto",
    "prune", "pumpkin", "punch", "quail", "quiche", "quince", "quinoa",
    "rabbit", "rack of lamb", "radicchio", "radish", "raisin", "raspberry",
    "red wine", "rhubarb", "rice", "ricotta", "rosé", "rum", "rutabaga", "rye",
    "saffron", "sage", "sake", "salad", "salad dressing", "salmon", "salsa",
    "sardine", "sauce", "sausage", "scallop", "scotch", "seafood", "seed",
    "semolina", "sesame", "sesame oil", "shallot", "shellfish", "sherry",
    "shrimp", "smoothie", "snapper", "sorbet", "sour cream", "sourdough",
    "soy", "soy sauce", "sparkling wine", "spice", "spinach", "squash",
    "squid", "steak", "stock", "strawberry", "sugar snap pea",
    "sweet potato/yam", "swiss cheese", "swordfish", "tamarind", "tangerine",
    "tapioca", "tarragon", "tart", "tea", "tequila", "thyme", "tilapia",
    "tofu", "tomatillo", "tomato", "tortillas", "triple sec", "tropical fruit",
    "trout", "tuna", "turnip", "vanilla", "veal", "venison", "vermouth",
    "vinegar", "vodka", "waffle", "walnut", "wasabi", "watercress",
    "watermelon", "whiskey", "white wine", "whole wheat", "wild rice", "wine",
    "yellow squash", "yogurt", "yuca", "zucchini", "turkey"
]

len(full_list_of_ingredients)

344

Здесь находятся битые названия ингредиентов, и такие, по которым в дальнейшем будет сложно осуществлять парсинг.
Они будут заменены на нормальные

In [4]:
replacements = {
    "butterscotch/caramel": "caramel",
    "jam or jelly": "jam",
    "cognac/armagnac": "cognac",
    "green onion/scallion": "scallion",
    "hominy/cornmeal/masa": "cornmeal",
    "milk/cream": "milk",
    "phyllo/puff pastry dough": "puff pastry dough",
    "sweet potato/yam": "sweet potato",
    "cr��me de cacao": "créme de cacao"
}

Напишем класс, который обработает датафрейм, уберет лишние колонки и отформатирует неправильные и соберет датафрейм необходимый для обучения модели

In [5]:
class IngredientPreprocessor:
    def __init__(self, ingredient_columns, replacements, target_column):
        self.ingredient_columns = ingredient_columns
        self.replacements = replacements
        self.target_column = target_column

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        df = X.copy()
        df = df[self.ingredient_columns + [self.target_column]].copy()

        for old_col, new_col in self.replacements.items():
            if old_col in df.columns:
                if new_col in df.columns:
                    df[new_col] = df[[old_col, new_col]].max(axis=1)
                else:
                    df[new_col] = df[old_col]
                df = df.drop(columns=[old_col])

        feature_cols = [col for col in df.columns if col != self.target_column]
        feature_cols = sorted(feature_cols)

        df = df[feature_cols + [self.target_column]]

        return df

    def fit_transform(self, X, y=None):
        return self.fit(X, y).transform(X)

Формируем датафрейм с колонками ингредиентов и целевой колонкой "rating"

In [6]:
preprocessor = IngredientPreprocessor(
    ingredient_columns=full_list_of_ingredients,
    replacements=replacements,
    target_column="rating"
)

preprocessing = Pipeline([
    ("ingredient_preprocessor", preprocessor)
])

final_df = preprocessor.fit_transform(df)

final_df

,almond,amaretto,anchovy,anise,apple,apple juice,apricot,artichoke,arugula,asian pear,...,whiskey,white wine,whole wheat,wild rice,wine,yellow squash,yogurt,yuca,zucchini,rating
0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.500
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.375
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.750
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,5.000
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.125
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20047,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.125
20048,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.375
20049,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.375
20050,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.375


Также добавим к этому датафрейму еще три колонки, которые не будут использованы при обучении моделей, но пригодятся нам в будущем для выполнения бонусной части задания

In [12]:
final_df = pd.concat([final_df, df[["breakfast", "lunch", "dinner"]]], axis=1)

Проверим, что в нашем датафрейме есть именно ингредиенты и не добавлены лишние колонки

In [14]:
cols = full_list_of_ingredients

checks = {
    "title": "not",
    "leftovers": "not",
    "california": "not",
    "dominican republic": "not",
    "protein": "not",
    "low cholesterol": "not",
    "pan-fry": "not",
    "fruit": "not",
    "stew": "not",
    "braise": "not",
    "milk": "in",
    "jam": "in"
}

for col, rule in checks.items():
    if rule == "not":
        print(col, col not in final_df.columns)
    else:
        print(col, col in final_df.columns)

title True
leftovers True
california True
dominican republic True
protein True
low cholesterol True
pan-fry True
fruit True
stew True
braise True
milk True
jam True


Формируем подвыборки

In [15]:
X = final_df.drop(columns=["rating", "breakfast", "lunch", "dinner"])
y = final_df["rating"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=21
)

### b. Regression

Выполним наивное предсказание и посчитаем RMSE

In [16]:
dummy_reg = DummyRegressor(strategy="mean")

dummy_reg.fit(X_train, y_train)
y_pred = dummy_reg.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
print("Naive RMSE:", rmse)

Naive RMSE: 1.307237065771332


Напишем функцию для подбора регрессионной модели и выполним подбор

In [17]:
def reg_GS_CV(model, param_grid, X_train, y_train, X_test, y_test):
    grid_search = GridSearchCV(
        estimator=model,
        param_grid=param_grid,
        cv=5,
        scoring="neg_root_mean_squared_error"
    )

    grid_search.fit(X_train, y_train)
    best_model = grid_search.best_estimator_
    y_pred = best_model.predict(X_test)

    test_rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    model_name = type(model).__name__

    print(
        f"\n{model_name}"
        f"\nBest params: {grid_search.best_params_}"
        f"\nTest RMSE: {test_rmse:.4f}"
    )

    return best_model

In [18]:
lr_reg_grid = reg_GS_CV(
    LinearRegression(),
    {},
    X_train, y_train, X_test, y_test
)

tree_reg_grid = reg_GS_CV(
    DecisionTreeRegressor(random_state=21),
    {
        "max_depth": [3, 5, 7, 9, None],
        "min_samples_split": [2, 5, 10],
        "min_samples_leaf": [1, 2, 5]
    },
    X_train, y_train, X_test, y_test
)

rf_reg_grid = reg_GS_CV(
    RandomForestRegressor(random_state=21),
    {
        "n_estimators": [50, 100],
        "max_depth": [10, 25, None],
        "min_samples_split": [2, 5],
        "min_samples_leaf": [2, 5]
    },
    X_train, y_train, X_test, y_test
)

gb_reg_grid = reg_GS_CV(
    GradientBoostingRegressor(random_state=21),
    {
        "n_estimators": [50, 100],
        "learning_rate": [0.05, 0.1],
        "max_depth": [2, 3, 5],
        "min_samples_split": [2, 5],
        "min_samples_leaf": [1, 2]
    },
    X_train, y_train, X_test, y_test
)

estimators = [
    ("lr", lr_reg_grid),
    ("rf", rf_reg_grid),
    ("gb", gb_reg_grid)
]

vr_reg_grid = reg_GS_CV(
    VotingRegressor(estimators=estimators),
    {
        "weights": list(product([1, 2], repeat=3))
    },
    X_train, y_train, X_test, y_test
)

br_reg_grid = reg_GS_CV(
    BaggingRegressor(
        tree_reg_grid,
        random_state=21
    ),
    {
        "n_estimators": [50, 100]
    },
    X_train, y_train, X_test, y_test
)

sr_reg_grid = reg_GS_CV(
    StackingRegressor(
        estimators=estimators,
        final_estimator=LinearRegression()
    ),
    {
        "passthrough": [True, False]
    },
    X_train, y_train, X_test, y_test
)


LinearRegression
Best params: {}
Test RMSE: 1.2668

DecisionTreeRegressor
Best params: {'max_depth': 9, 'min_samples_leaf': 5, 'min_samples_split': 2}
Test RMSE: 1.2903

RandomForestRegressor
Best params: {'max_depth': None, 'min_samples_leaf': 5, 'min_samples_split': 2, 'n_estimators': 100}
Test RMSE: 1.2608

GradientBoostingRegressor
Best params: {'learning_rate': 0.1, 'max_depth': 5, 'min_samples_leaf': 2, 'min_samples_split': 5, 'n_estimators': 100}
Test RMSE: 1.2660

VotingRegressor
Best params: {'weights': (1, 2, 1)}
Test RMSE: 1.2517

BaggingRegressor
Best params: {'n_estimators': 100}
Test RMSE: 1.2754

StackingRegressor
Best params: {'passthrough': False}
Test RMSE: 1.2526


### b. Classification

#### Binary Classification

Напишем функцию для подбора модели классификации, выполним подбор и посчитаем accuracy

In [19]:
def class_GS_CV(model, param_grid,
                X_train, y_train,
                X_test, y_test,
                scoring="accuracy"):
    grid_search = GridSearchCV(
        estimator=model,
        param_grid=param_grid,
        cv=3,
        scoring=scoring
    )

    grid_search.fit(X_train, y_train)
    best_model = grid_search.best_estimator_
    y_pred = best_model.predict(X_test)

    acc = accuracy_score(y_test, y_pred)

    model_name = type(model).__name__

    print(
        f"\n{model_name}"
        f"\nBest params: {grid_search.best_params_}"
        f"\nAccuracy: {acc:.4f}"
    )

    return best_model

In [20]:
y_train_class = np.round(y_train).astype(int)
y_test_class = np.round(y_test).astype(int)

In [21]:
dummy_class = DummyClassifier(strategy="most_frequent")

dummy_class.fit(X_train, y_train_class)
y_pred_class = dummy_class.predict(X_test)

accuracy = accuracy_score(y_test_class, y_pred_class)
print(f"Naive accuracy: {accuracy:.4f}")

Naive accuracy: 0.6707


In [22]:
lr_class_grid = class_GS_CV(
    LogisticRegression(fit_intercept=False, max_iter=5000),
    {
        "C": [0.1, 1, 10]
    },
    X_train, y_train_class, X_test, y_test_class
)

tree_class_grid = class_GS_CV(
    DecisionTreeClassifier(random_state=21),
    {
        "max_depth": [5, 10, None],
        "min_samples_split": [2, 5]
    },
    X_train, y_train_class, X_test, y_test_class
)

rf_class_grid = class_GS_CV(
    RandomForestClassifier(random_state=21),
    {
        "n_estimators": [50, 100],
        "max_depth": [10, None]
    },
    X_train, y_train_class, X_test, y_test_class
)

gb_class_grid = class_GS_CV(
    GradientBoostingClassifier(random_state=21),
    {
        "n_estimators": [50, 100],
        "learning_rate": [0.05, 0.1]
    },
    X_train, y_train_class, X_test, y_test_class
)


LogisticRegression
Best params: {'C': 0.1}
Accuracy: 0.6662

DecisionTreeClassifier
Best params: {'max_depth': 10, 'min_samples_split': 2}
Accuracy: 0.6749

RandomForestClassifier
Best params: {'max_depth': 10, 'n_estimators': 100}
Accuracy: 0.6731

GradientBoostingClassifier
Best params: {'learning_rate': 0.05, 'n_estimators': 100}
Accuracy: 0.6769


#### Bad, so-so, great

Превратим числовой рейтинг в три смысловых класса great, so-so, bad в зависимости от оценки блюда и повторим процедуру

In [23]:
BINS = [-1, 1, 3, 5]
LABELS = ["bad", "so-so", "great"]

y_train_alt = pd.cut(y_train.round().astype(int), bins=BINS, labels=LABELS)
y_test_alt = pd.cut(y_test.round().astype(int), bins=BINS, labels=LABELS)

In [24]:
dummy_alt = DummyClassifier(strategy="most_frequent")

dummy_alt.fit(X_train, y_train_alt)
y_pred_alt = dummy_alt.predict(X_test)

accuracy = accuracy_score(y_test_alt, y_pred_alt)
print(f"Naive accuracy: {accuracy:.4f}")

Naive accuracy: 0.7978


In [25]:
lr_alt_grid = class_GS_CV(
    LogisticRegression(fit_intercept=False, max_iter=5000),
    {
        "C": [0.1, 1, 10]
    },
    X_train, y_train_alt, X_test, y_test_alt
)

tree_alt_grid = class_GS_CV(
    DecisionTreeClassifier(random_state=21),
    {
        "max_depth": [5, 10, None],
        "min_samples_split": [2, 5]
    },
    X_train, y_train_alt, X_test, y_test_alt
)

rf_alt_grid = class_GS_CV(
    RandomForestClassifier(random_state=21),
    {
        "n_estimators": [50, 100],
        "max_depth": [10, None]
    },
    X_train, y_train_alt, X_test, y_test_alt
)

gb_alt_grid = class_GS_CV(
    GradientBoostingClassifier(random_state=21),
    {
        "n_estimators": [50, 100],
        "learning_rate": [0.05, 0.1]
    },
    X_train, y_train_alt, X_test, y_test_alt
)


LogisticRegression
Best params: {'C': 0.1}
Accuracy: 0.7911

DecisionTreeClassifier
Best params: {'max_depth': 10, 'min_samples_split': 2}
Accuracy: 0.7996

RandomForestClassifier
Best params: {'max_depth': 10, 'n_estimators': 50}
Accuracy: 0.7988

GradientBoostingClassifier
Best params: {'learning_rate': 0.1, 'n_estimators': 100}
Accuracy: 0.8010


#### Comparing the metric

Предсказывать рецепт как "great", когда на самом деле он "bad", хуже, чем пропустить действительно хороший рецепт.  
Это означает, что для нас критичны ошибки типа false positive для класса "great".  
Метрика precision для класса "great" показывает долю корректных предсказаний среди всех предсказаний "great":

$$
Precision = \frac{TP}{TP + FP}
$$

Таким образом, использование precision позволяет минимизировать количество ложных рекомендаций хороших рецептов.

In [26]:
def great_precision(y_true, y_pred):
    precision = precision_score(
        y_true, y_pred,
        labels=["great"],
        average="macro",
        zero_division=0
    )

    return precision

precision_great_scorer = make_scorer(great_precision)

class_GS_CV_prec = partial(class_GS_CV, scoring=precision_great_scorer)

In [27]:
lr_alt_grid_prec = class_GS_CV_prec(
    LogisticRegression(fit_intercept=False, max_iter=5000),
    {
        "C": [0.1, 1, 10]
    },
    X_train, y_train_alt, X_test, y_test_alt
)

tree_alt_grid_prec = class_GS_CV_prec(
    DecisionTreeClassifier(random_state=21),
    {
        "max_depth": [5, 10, None],
        "min_samples_split": [2, 5]
    },
    X_train, y_train_alt, X_test, y_test_alt
)

rf_alt_grid_prec = class_GS_CV_prec(
    RandomForestClassifier(random_state=21),
    {
        "n_estimators": [50, 100],
        "max_depth": [10, None]
    },
    X_train, y_train_alt, X_test, y_test_alt
)

gb_alt_grid_prec = class_GS_CV_prec(
    GradientBoostingClassifier(random_state=21),
    {
        "n_estimators": [50, 100],
        "learning_rate": [0.05, 0.1]
    },
    X_train, y_train_alt, X_test, y_test_alt
)

estimators = [
    ("lr", lr_alt_grid_prec),
    ("rf", rf_alt_grid_prec),
    ("gb", gb_alt_grid_prec)
]

vc_alt_grid_prec = class_GS_CV_prec(
    VotingClassifier(estimators=estimators),
    {
        "weights": list(product([1, 2], repeat=3))
    },
    X_train, y_train_alt, X_test, y_test_alt
)

bc_alt_grid_prec = class_GS_CV_prec(
    BaggingClassifier(
        tree_alt_grid_prec,
        random_state=21
    ),
    {
        "n_estimators": [50, 100]
    },
    X_train, y_train_alt, X_test, y_test_alt
)

sc_alt_grid_prec = class_GS_CV_prec(
    StackingClassifier(
        estimators=estimators,
        final_estimator=LogisticRegression(fit_intercept=False, max_iter=5000)
    ),
    {
        "passthrough": [True, False]
    },
    X_train, y_train_alt, X_test, y_test_alt
)


LogisticRegression
Best params: {'C': 10}
Accuracy: 0.7916

DecisionTreeClassifier
Best params: {'max_depth': None, 'min_samples_split': 2}
Accuracy: 0.7098

RandomForestClassifier
Best params: {'max_depth': None, 'n_estimators': 50}
Accuracy: 0.7906

GradientBoostingClassifier
Best params: {'learning_rate': 0.1, 'n_estimators': 100}
Accuracy: 0.8010

VotingClassifier
Best params: {'weights': (1, 2, 1)}
Accuracy: 0.7941

BaggingClassifier
Best params: {'n_estimators': 100}
Accuracy: 0.7709

StackingClassifier
Best params: {'passthrough': True}
Accuracy: 0.8050


Поскольку итоговая программа должна вывести один из трех классов (“плохой”, “так себе”, “отличный”), я решил использовать классификационный подход. Он в большей степени соответствует требованиям к конечному продукту, чем регрессия, поскольку позволяет напрямую прогнозировать целевые классы, а не требует дополнительного преобразования числового рейтинга.

In [28]:
best_model = sc_alt_grid_prec

joblib.dump(best_model, "./data/best_model.pkl")

['./data/best_model.pkl']

## 2. Nutrition facts

Загрузим ключ API необходимый для доступа к данным об ингредиентах, также загрузим id ингредиентов для считывания именно их данных, а не первых попавшихся, и информацию о суточной норме питательных веществ

In [29]:
load_dotenv()
API_KEY = os.getenv("FOODS_API")

BASE_URL = "https://api.nal.usda.gov/fdc/v1"

ids_df = pd.read_csv("./data/ingredient_fdc_ids.csv")

daily_values = {
    "Protein": {"value": 50, "unit": "g"},
    "Total lipid (fat)": {"value": 78, "unit": "g"},
    "Fatty acids, total saturated": {"value": 20, "unit": "g"},
    "Cholesterol": {"value": 300, "unit": "mg"},
    "Carbohydrate, by difference": {"value": 275, "unit": "g"},
    "Sodium, Na": {"value": 2300, "unit": "mg"},
    "Fiber, total dietary": {"value": 28, "unit": "g"},
    "Sugars, added": {"value": 50, "unit": "g"},
    "Vitamin A, RAE": {"value": 900, "unit": "mcg"},
    "Vitamin C, total ascorbic acid": {"value": 90, "unit": "mg"},
    "Calcium, Ca": {"value": 1300, "unit": "mg"},
    "Iron, Fe": {"value": 18, "unit": "mg"},
    "Vitamin D (D2 + D3)": {"value": 20, "unit": "mcg"},
    "Vitamin E (alpha-tocopherol)": {"value": 15, "unit": "mg"},
    "Vitamin K (phylloquinone)": {"value": 120, "unit": "mcg"},
    "Thiamin": {"value": 1.2, "unit": "mg"},
    "Riboflavin": {"value": 1.3, "unit": "mg"},
    "Niacin": {"value": 16, "unit": "mg"},
    "Vitamin B-6": {"value": 1.7, "unit": "mg"},
    "Folate, DFE": {"value": 400, "unit": "mcg"},
    "Vitamin B-12": {"value": 2.4, "unit": "mcg"},
    "Biotin": {"value": 30, "unit": "mcg"},
    "Pantothenic acid": {"value": 5, "unit": "mg"},
    "Phosphorus, P": {"value": 1250, "unit": "mg"},
    "Iodine, I": {"value": 150, "unit": "mcg"},
    "Magnesium, Mg": {"value": 420, "unit": "mg"},
    "Zinc, Zn": {"value": 11, "unit": "mg"},
    "Selenium, Se": {"value": 55, "unit": "mcg"},
    "Copper, Cu": {"value": 0.9, "unit": "mg"},
    "Manganese, Mn": {"value": 2.3, "unit": "mg"},
    "Chromium, Cr": {"value": 35, "unit": "mcg"},
    "Molybdenum, Mo": {"value": 45, "unit": "mcg"},
    "Chloride, Cl": {"value": 2300, "unit": "mg"},
    "Potassium, K": {"value": 4700, "unit": "mg"},
    "Choline, total": {"value": 550, "unit": "mg"}
}

Напишем функции для считывания данных с сервиса и перевода их в процент от суточной нормы

In [30]:
def convert_unit(value, from_unit, to_unit):
    if pd.isna(value) or from_unit is None or to_unit is None:
        return np.nan

    to_mg = {
        "G": 1000.0,
        "MG": 1.0,
        "MCG": 0.001,
        "UG": 0.001,
    }

    from_unit = from_unit.upper()
    to_unit = to_unit.upper()

    if from_unit not in to_mg or to_unit not in to_mg:
        return np.nan

    value_mg = value * to_mg[from_unit]
    return value_mg / to_mg[to_unit]

def get_food_details(fdc_id, retries=3, pause=2):
    url = f"{BASE_URL}/food/{fdc_id}"
    params = {"api_key": API_KEY}

    for attempt in range(retries):
        try:
            response = requests.get(url, params=params, timeout=60)
            response.raise_for_status()
            return response.json()
        except requests.exceptions.ReadTimeout:
            print(f"Timeout for {fdc_id}, attempt {attempt + 1}/{retries}")
            time.sleep(pause)

    raise RuntimeError(f"Failed to fetch {fdc_id} after {retries} attempts")

def extract_percent_dv(food_details, daily_values):
    result = {nutrient_name: 0 for nutrient_name in daily_values.keys()}

    for item in food_details.get("foodNutrients", []):
        nutrient = item.get("nutrient", {})
        nutrient_name = nutrient.get("name")
        nutrient_unit = nutrient.get("unitName")
        amount = item.get("amount")

        is_known_nutrient = nutrient_name in daily_values
        has_amount = amount is not None
        has_unit = nutrient_unit is not None

        if is_known_nutrient and has_amount and has_unit:
            dv_value = daily_values[nutrient_name]["value"]
            dv_unit = daily_values[nutrient_name]["unit"]

            converted_amount = convert_unit(
                value=amount,
                from_unit=nutrient_unit,
                to_unit=dv_unit
            )

            if not pd.isna(converted_amount):
                percent = converted_amount / dv_value * 100
                result[nutrient_name] = round(percent, 2)

    return result

Выполним считывание данных

In [31]:
rows = []

for _, row in ids_df.iterrows():
    ingredient = row["ingredient"]
    fdc_id = row["fdc_id"]

    try:
        food_details = get_food_details(fdc_id)
        percent_dv = extract_percent_dv(food_details, daily_values)

        result_row = {
            "ingredient": ingredient,
            "fdc_id": fdc_id,
            "description": food_details.get("description"),
            **percent_dv
        }
        rows.append(result_row)

        time.sleep(0.1)

    except Exception as e:
        print(f"Error for {ingredient} ({fdc_id}): {e}")

        result_row = {
            "ingredient": ingredient,
            "fdc_id": fdc_id,
            "description": None,
            **{nutrient_name: 0 for nutrient_name in daily_values.keys()}
        }
        rows.append(result_row)

nutrition_df = pd.DataFrame(rows)
nutrition_df = nutrition_df.set_index("ingredient")

numeric_columns = [
    col
    for col in nutrition_df.columns
    if col not in ["fdc_id", "description"]
]

nutrition_df[numeric_columns] = nutrition_df[numeric_columns].fillna(0)

nutrition_df.to_csv("./data/nutrition_data_by_fdc_id.csv")

## 3. Similar Recipes

Теперь для каждого рецепта найдем ссылки

In [ ]:
BASE_URL = "https://www.epicurious.com"

df = pd.read_csv("./data/epi_r.csv")

rows = []

for title in df["title"].drop_duplicates():
    try:
        response = requests.get(f"{BASE_URL}/search", params={"q": title})
        soup = BeautifulSoup(response.text, "html.parser")

        url = None
        for a in soup.find_all("a", href=True):
            if a["href"].startswith("/recipes/food/views/"):
                url = BASE_URL + a["href"]
                break

        if url:
            rows.append({"title": title, "url": url})

        time.sleep(0.1)

    except Exception as e:
        print(f"Error for {title}: {e}")
        continue

Сохраним информацию о рецептах вместе с ссылками

In [ ]:
recipes_urls_df = pd.DataFrame(rows)

result_df = pd.concat(
    [recipes_urls_df.reset_index(drop=True),
     final_df.reset_index(drop=True)],
    axis=1
)

result_df = result_df.dropna(axis=0, how='any')

result_df.to_csv("./data/recipes_with_urls.csv", index=False)